# Network KPI Anomaly Detection
**Stack:** PyTorch · Scikit-learn · Pandas · Matplotlib  
**Dataset:** Alibaba/Tsinghua KPI Anomaly Detection Benchmark (real telecom operator data)  
**Approach:** Baseline (Isolation Forest) vs Deep (LSTM Autoencoder) — comparative analysis

---

## 1. Setup & Data Download

In [ ]:
# Install dependencies
!pip install -q gdown scikit-learn torch pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# Download the KPI Anomaly Detection Benchmark
# Source: https://github.com/NetManAIOps/KPI-Anomaly-Detection
!git clone --quiet https://github.com/NetManAIOps/KPI-Anomaly-Detection.git

import os, glob
train_path = 'KPI-Anomaly-Detection/Finals_dataset/phase2_train.csv'
test_path  = 'KPI-Anomaly-Detection/Finals_dataset/phase2_ground_truth.hdf'

# Load training data
df_train = pd.read_csv(train_path)
print('Train shape:', df_train.shape)
print(df_train.head())
print('\nColumns:', df_train.columns.tolist())
print('\nLabel distribution:')
print(df_train['label'].value_counts())

## 2. Exploratory Data Analysis

In [ ]:
# Pick one KPI series to work with (can loop over all later)
kpi_ids = df_train['KPI ID'].unique()
print(f'Total KPI series: {len(kpi_ids)}')

# Select a KPI with a good mix of anomalies
selected_kpi = None
for kpi in kpi_ids:
    subset = df_train[df_train['KPI ID'] == kpi]
    anomaly_rate = subset['label'].mean()
    if 0.02 < anomaly_rate < 0.15:  # 2-15% anomaly rate is realistic
        selected_kpi = kpi
        print(f'Selected KPI: {kpi} | Anomaly rate: {anomaly_rate:.2%} | Length: {len(subset)}')
        break

kpi_df = df_train[df_train['KPI ID'] == selected_kpi].sort_values('timestamp').reset_index(drop=True)

# Plot the raw series
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(kpi_df['value'], color='steelblue', lw=0.8, label='KPI value')
anomaly_idx = kpi_df[kpi_df['label'] == 1].index
ax.scatter(anomaly_idx, kpi_df.loc[anomaly_idx, 'value'],
           color='red', s=10, zorder=5, label='Ground truth anomaly')
ax.set_title('Raw Network KPI Time-Series with Ground Truth Anomalies', fontsize=13)
ax.legend()
ax.set_xlabel('Timestep')
ax.set_ylabel('KPI Value')
plt.tight_layout()
plt.savefig('01_raw_kpi.png', dpi=150)
plt.show()
print(f'Ground truth anomalies: {kpi_df["label"].sum()} / {len(kpi_df)} points')

## 3. Preprocessing

In [ ]:
def preprocess(df, window_size=60):
    """
    - Normalise values to [0, 1]
    - Create sliding windows of shape (N, window_size, 1)
    - Return windows, labels (per-window: 1 if any anomaly in window)
    """
    scaler = MinMaxScaler()
    values = scaler.fit_transform(df[['value']]).flatten()
    labels = df['label'].values

    X, y = [], []
    for i in range(len(values) - window_size):
        X.append(values[i:i+window_size])
        # Label a window anomalous if the LAST point is anomalous (predictive framing)
        y.append(labels[i + window_size - 1])

    return np.array(X), np.array(y), scaler


WINDOW_SIZE = 60   # 60 timesteps per window (~1 hour if 1-min resolution)
TRAIN_SPLIT = 0.7

X, y, scaler = preprocess(kpi_df, window_size=WINDOW_SIZE)

split = int(len(X) * TRAIN_SPLIT)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Only use normal data for autoencoder training (unsupervised anomaly detection)
X_train_normal = X_train[y_train == 0]

print(f'Train windows: {len(X_train)} | Normal only: {len(X_train_normal)}')
print(f'Test windows:  {len(X_test)} | Anomalies: {y_test.sum()} ({y_test.mean():.2%})')

## 4. Baseline — Isolation Forest

In [ ]:
print('Training Isolation Forest...')

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=float(y_train.mean()),  # use known anomaly rate from train set
    random_state=42,
    n_jobs=-1
)

# Train on ALL train windows (IF handles unsupervised detection)
iso_forest.fit(X_train)

# Predict: IF returns -1 for anomaly, 1 for normal → remap to 0/1
iso_preds_raw = iso_forest.predict(X_test)
iso_preds = (iso_preds_raw == -1).astype(int)

# Anomaly scores (lower = more anomalous)
iso_scores = -iso_forest.decision_function(X_test)  # negate so higher = more anomalous

print('\n--- Isolation Forest Results ---')
print(classification_report(y_test, iso_preds, target_names=['Normal', 'Anomaly']))
print(f'ROC-AUC: {roc_auc_score(y_test, iso_scores):.4f}')

iso_results = {
    'f1':        f1_score(y_test, iso_preds),
    'precision': precision_score(y_test, iso_preds),
    'recall':    recall_score(y_test, iso_preds),
    'roc_auc':   roc_auc_score(y_test, iso_scores)
}

## 5. Deep Model — LSTM Autoencoder

In [ ]:
class LSTMAutoencoder(nn.Module):
    """
    Encoder: compress sequence to latent vector
    Decoder: reconstruct sequence from latent vector
    Anomaly score = reconstruction error (MSE)
    High error → pattern not seen during normal training → likely anomaly
    """
    def __init__(self, input_size=1, hidden_size=64, latent_size=16, num_layers=2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # Encoder
        self.encoder = nn.LSTM(input_size, hidden_size,
                               num_layers=num_layers, batch_first=True, dropout=0.2)
        self.enc_fc  = nn.Linear(hidden_size, latent_size)

        # Decoder
        self.dec_fc  = nn.Linear(latent_size, hidden_size)
        self.decoder = nn.LSTM(hidden_size, hidden_size,
                               num_layers=num_layers, batch_first=True, dropout=0.2)
        self.output  = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        # x: (batch, seq_len, 1)
        batch_size, seq_len, _ = x.shape

        # Encode
        _, (h, _) = self.encoder(x)
        latent    = self.enc_fc(h[-1])               # (batch, latent_size)

        # Decode — repeat latent across time steps
        dec_input = self.dec_fc(latent).unsqueeze(1).repeat(1, seq_len, 1)  # (batch, seq_len, hidden)
        dec_out, _ = self.decoder(dec_input)
        recon      = self.output(dec_out)            # (batch, seq_len, 1)

        return recon

In [ ]:
# Prepare DataLoaders
def make_loader(X, batch_size=64, shuffle=True):
    t = torch.FloatTensor(X).unsqueeze(-1)  # (N, seq_len, 1)
    return DataLoader(TensorDataset(t, t), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train_normal, batch_size=64, shuffle=True)
test_loader  = make_loader(X_test,         batch_size=64, shuffle=False)

# Model, optimizer, loss
model     = LSTMAutoencoder(input_size=1, hidden_size=64, latent_size=16, num_layers=2).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Training on: {DEVICE}')

In [ ]:
# Training loop
EPOCHS = 40
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for xb, _ in train_loader:
        xb = xb.to(DEVICE)
        optimizer.zero_grad()
        recon = model(xb)
        loss  = criterion(recon, xb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    scheduler.step(avg_loss)

    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg_loss:.6f} | LR: {optimizer.param_groups[0]["lr"]:.2e}')

# Plot training curve
plt.figure(figsize=(8, 3))
plt.plot(train_losses, color='steelblue')
plt.title('LSTM Autoencoder — Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.tight_layout()
plt.savefig('02_training_loss.png', dpi=150)
plt.show()

## 6. Threshold Sweep & Evaluation

In [ ]:
# Compute reconstruction errors on test set
model.eval()
recon_errors = []

with torch.no_grad():
    for xb, _ in test_loader:
        xb   = xb.to(DEVICE)
        recon = model(xb)
        mse   = ((recon - xb) ** 2).mean(dim=(1, 2))  # per-sample MSE
        recon_errors.extend(mse.cpu().numpy())

recon_errors = np.array(recon_errors)

# Threshold sweep: find best F1
thresholds = np.percentile(recon_errors, np.linspace(80, 99, 50))
best_f1, best_thresh, best_preds = 0, 0, None
f1_curve = []

for thresh in thresholds:
    preds = (recon_errors > thresh).astype(int)
    f1    = f1_score(y_test, preds, zero_division=0)
    f1_curve.append(f1)
    if f1 > best_f1:
        best_f1     = f1
        best_thresh = thresh
        best_preds  = preds

print(f'Best threshold: {best_thresh:.6f}')
print(f'Best F1:        {best_f1:.4f}')
print('\n--- LSTM Autoencoder Results ---')
print(classification_report(y_test, best_preds, target_names=['Normal', 'Anomaly']))
print(f'ROC-AUC: {roc_auc_score(y_test, recon_errors):.4f}')

lstm_results = {
    'f1':        best_f1,
    'precision': precision_score(y_test, best_preds, zero_division=0),
    'recall':    recall_score(y_test, best_preds, zero_division=0),
    'roc_auc':   roc_auc_score(y_test, recon_errors)
}

In [ ]:
# Plot threshold sweep
plt.figure(figsize=(8, 3))
plt.plot(thresholds, f1_curve, color='steelblue')
plt.axvline(best_thresh, color='red', linestyle='--', label=f'Best threshold (F1={best_f1:.3f})')
plt.title('F1 Score vs Reconstruction Error Threshold')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.legend()
plt.tight_layout()
plt.savefig('03_threshold_sweep.png', dpi=150)
plt.show()

## 7. Anomaly Visualisation

In [ ]:
test_values = kpi_df['value'].values[int(len(kpi_df)*TRAIN_SPLIT) + WINDOW_SIZE:]
n_plot = min(len(test_values), len(recon_errors))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 6), sharex=True)

# Top: raw signal with detected anomalies
ax1.plot(test_values[:n_plot], color='steelblue', lw=0.8, label='KPI value')
gt_idx   = np.where(y_test[:n_plot] == 1)[0]
pred_idx = np.where(best_preds[:n_plot] == 1)[0]
ax1.scatter(gt_idx,   test_values[gt_idx],   color='red',    s=12, zorder=5, label='Ground truth')
ax1.scatter(pred_idx, test_values[pred_idx], color='orange', s=8,  zorder=4, marker='^', label='LSTM predicted')
ax1.set_title('Network KPI — Ground Truth vs LSTM Autoencoder Detections')
ax1.legend(fontsize=9)
ax1.set_ylabel('KPI Value')

# Bottom: reconstruction error
ax2.plot(recon_errors[:n_plot], color='purple', lw=0.7, label='Reconstruction error')
ax2.axhline(best_thresh, color='red', linestyle='--', lw=1, label=f'Threshold = {best_thresh:.4f}')
ax2.fill_between(range(n_plot), recon_errors[:n_plot], best_thresh,
                 where=recon_errors[:n_plot] > best_thresh,
                 alpha=0.3, color='red', label='Anomaly region')
ax2.set_title('Reconstruction Error over Time')
ax2.set_xlabel('Timestep')
ax2.set_ylabel('MSE')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('04_anomaly_detection.png', dpi=150)
plt.show()

## 8. Comparative Analysis — IF vs LSTM Autoencoder

In [ ]:
metrics = ['f1', 'precision', 'recall', 'roc_auc']
labels  = ['F1 Score', 'Precision', 'Recall', 'ROC-AUC']

iso_vals  = [iso_results[m]  for m in metrics]
lstm_vals = [lstm_results[m] for m in metrics]

x     = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
bars1 = ax.bar(x - width/2, iso_vals,  width, label='Isolation Forest', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, lstm_vals, width, label='LSTM Autoencoder', color='coral',     alpha=0.85)

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f'{h:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.12)
ax.set_title('Model Comparison — Isolation Forest vs LSTM Autoencoder')
ax.set_ylabel('Score')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('05_comparison.png', dpi=150)
plt.show()

print('\n=== FINAL SUMMARY ===')
print(f'{"Metric":<12} {"Isolation Forest":>18} {"LSTM Autoencoder":>18}')
print('-' * 50)
for m, l in zip(metrics, labels):
    print(f'{l:<12} {iso_results[m]:>18.4f} {lstm_results[m]:>18.4f}')

## 9. Key Takeaways

| | Isolation Forest | LSTM Autoencoder |
|---|---|---|
| **Training** | Seconds | Minutes (GPU) |
| **Temporal context** | No (treats each window as flat vector) | Yes (learns sequential patterns) |
| **Interpretability** | Anomaly score only | Reconstruction error + per-timestep |
| **Best for** | Quick baseline, low-latency inference | Complex temporal patterns, higher accuracy |
| **Production use** | Edge devices, resource-constrained | Cloud/server-side network monitoring |

**Key insight:** LSTM Autoencoder learns the *shape* of normal network behaviour. When a KPI exhibits an unusual pattern (spike, level shift, oscillation), the reconstruction error spikes — providing both detection and an implicit explanation of *what* changed.

**Extensibility:** This architecture directly applies to real 5G RAN monitoring — swapping the KPI dataset for real base station telemetry (RSRP, SINR, PRB utilisation) requires minimal code changes.

In [ ]:
# Save model checkpoint
torch.save({
    'model_state_dict': model.state_dict(),
    'threshold':        best_thresh,
    'window_size':      WINDOW_SIZE,
    'lstm_results':     lstm_results,
    'iso_results':      iso_results
}, 'network_anomaly_model.pth')

print('Model checkpoint saved: network_anomaly_model.pth')
print('Plots saved: 01_raw_kpi.png, 02_training_loss.png, 03_threshold_sweep.png, 04_anomaly_detection.png, 05_comparison.png')